# 🍌 Banana Leaf Disease Detection — Multi-Model Ensemble

Training 3 model (Custom CNN, ResNet50, InceptionV3) dengan **Soft Voting Ensemble**.

**7 Kelas:**
1. Banana Black Sigatoka Disease
2. Banana Bract Mosaic Virus Disease
3. Banana Healthy Leaf
4. Banana Insect Pest Disease
5. Banana Moko Disease
6. Banana Panama Disease
7. Banana Yellow Sigatoka Disease

## 1. Setup: Kaggle API + Google Drive

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive (untuk simpan hasil training nanti)
from google.colab import drive
drive.mount('/content/drive')

### Download Dataset dari Kaggle

**Cara dapat `kaggle.json`:**
1. Buka https://www.kaggle.com/settings/api
2. Scroll ke bagian **Legacy API Credentials**
3. Klik **Create Legacy API Key** → file `kaggle.json` otomatis terdownload
4. Upload file tersebut di cell berikut

In [ ]:
import os
from google.colab import files

# Upload kaggle.json (Legacy API Key)
print('Upload file kaggle.json dari komputer kamu:')
uploaded = files.upload()

# Setup Kaggle credentials
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
os.rename('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('\n✅ Kaggle API key configured!')

In [ ]:
# Download dataset langsung dari Kaggle (server-to-server, cepat!)
!kaggle datasets download -d sujaykapadnis/banana-disease-recognition-dataset -p /content/dataset --unzip

# Cek isi folder
import os
print('\nStruktur folder dataset:')
for root, dirs, fls in os.walk('/content/dataset'):
    level = root.replace('/content/dataset', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:
        for d in sorted(dirs):
            print(f'{indent}  {d}/')
    if level >= 2:
        print(f'{indent}  ({len(fls)} files)')
        break

In [ ]:
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense, Dropout, BatchNormalization,
    GlobalAveragePooling2D
)
from tensorflow.keras.applications import ResNet50, InceptionV3
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix
from scipy.ndimage import sobel

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

## 2. Configuration

In [ ]:
# Path dataset (hasil download Kaggle)
# Jika error, uncomment baris !ls untuk cek struktur folder yang benar
# !ls /content/dataset/
# !ls '/content/dataset/Banana Disease Recognition Dataset/'
# !ls '/content/dataset/Banana Disease Recognition Dataset/Augmented images/'

DATA_DIR = '/content/dataset/Banana Disease Recognition Dataset/Augmented images/Augmented images'

# Output directory
OUTPUT_DIR = '/content/artifacts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hyperparameters
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50
LABEL_SMOOTHING = 0.05
NUM_CLASSES = 7

print(f'Dataset path: {DATA_DIR}')
print(f'Output path: {OUTPUT_DIR}')
print(f'Image size: {IMAGE_SIZE}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Max epochs per model: {EPOCHS}')

## 3. Data Loading + Augmentation

In [ ]:
# Data Augmentation untuk training (mengurangi overfitting)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation hanya rescale (tanpa augmentasi)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# Simpan labels
labels = list(train_generator.class_indices.keys())
print(f'\nKelas ditemukan ({len(labels)}): {labels}')
print(f'Training samples: {train_generator.samples}')
print(f'Validation samples: {val_generator.samples}')

In [ ]:
# Class Weights (dataset tidak seimbang)
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weights_dict = {i: class_weights[i] for i in range(len(class_weights))}

print('\nClass weights:')
for i, label in enumerate(labels):
    print(f'  {label}: {class_weights_dict[i]:.3f}')

## 4. Visualisasi Data

In [ ]:
# Visualisasi sample data
images, img_labels = next(train_generator)

fig, axes = plt.subplots(4, 8, figsize=(20, 10))
axes = axes.flatten()
for img, lbl, ax in zip(images, img_labels, axes):
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(labels[np.argmax(lbl)], fontsize=7)
plt.suptitle('Training Samples', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Sobel Edge Detection Visualization
def apply_sobel(images_arr):
    sobel_images = []
    for img in images_arr:
        gray_img = np.dot(img[...,:3], [0.2989, 0.5870, 0.1140])
        sobel_x = sobel(gray_img, axis=0, mode='constant')
        sobel_y = sobel(gray_img, axis=1, mode='constant')
        sobel_img = np.hypot(sobel_x, sobel_y)
        sobel_images.append(sobel_img)
    return np.array(sobel_images)

sobel_images = apply_sobel(images)

fig, axes = plt.subplots(4, 8, figsize=(20, 10))
axes = axes.flatten()
for sob_img, lbl, ax in zip(sobel_images, img_labels, axes):
    ax.imshow(sob_img, cmap='gray')
    ax.axis('off')
    ax.set_title(labels[np.argmax(lbl)], fontsize=7)
plt.suptitle('Sobel Edge Detection', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Helper Functions

In [ ]:
def get_callbacks(model_name):
    """Callbacks untuk setiap model: EarlyStopping, Checkpoint, ReduceLR."""
    return [
        EarlyStopping(
            monitor='val_loss',
            patience=7,
            restore_best_weights=True,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=os.path.join(OUTPUT_DIR, f'best_{model_name}.keras'),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.2,
            patience=3,
            min_lr=1e-7,
            verbose=1
        )
    ]


def plot_history(history, model_name):
    """Plot training history (accuracy & loss)."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history.history['accuracy'], label='Train Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
    ax1.set_title(f'{model_name} — Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history.history['loss'], label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Val Loss')
    ax2.set_title(f'{model_name} — Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'history_{model_name}.png'), dpi=150)
    plt.show()


def evaluate_model(model, generator, model_name):
    """Evaluasi model individual."""
    print(f'\n{"="*60}')
    print(f'Evaluasi: {model_name}')
    print(f'{"="*60}')

    y_true = generator.classes
    y_pred = np.argmax(model.predict(generator, verbose=0), axis=1)

    print(classification_report(y_true, y_pred, target_names=labels))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(f'Confusion Matrix — {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    plt.show()

    return y_pred

## 6. Model 1: Custom CNN

In [ ]:
model_cnn = Sequential([
    Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(*IMAGE_SIZE, 3)),
    BatchNormalization(),
    Conv2D(32, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),

    Conv2D(64, kernel_size=(3, 3), activation='relu'),
    BatchNormalization(),
    Conv2D(64, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),

    Conv2D(128, kernel_size=(3, 3), activation='relu'),
    BatchNormalization(),
    Conv2D(128, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),

    Conv2D(256, kernel_size=(3, 3), activation='relu'),
    BatchNormalization(),
    Conv2D(256, kernel_size=(3, 3), activation='relu'),
    GlobalAveragePooling2D(),
    Dropout(0.5),

    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),

    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),

    Dense(NUM_CLASSES, activation='softmax')
])

model_cnn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy']
)

model_cnn.summary()

In [ ]:
print('\n' + '='*60)
print('Training Model 1: Custom CNN')
print('='*60)

history_cnn = model_cnn.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    class_weight=class_weights_dict,
    callbacks=get_callbacks('cnn'),
    verbose=1
)

plot_history(history_cnn, 'Custom CNN')

## 7. Model 2: ResNet50

In [ ]:
base_model_resnet = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(*IMAGE_SIZE, 3)
)
base_model_resnet.trainable = False

model_resnet = Sequential([
    base_model_resnet,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

model_resnet.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy']
)

model_resnet.summary()

In [ ]:
print('\n' + '='*60)
print('Training Model 2: ResNet50')
print('='*60)

history_resnet = model_resnet.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    class_weight=class_weights_dict,
    callbacks=get_callbacks('resnet'),
    verbose=1
)

plot_history(history_resnet, 'ResNet50')

## 8. Model 3: InceptionV3

In [ ]:
base_model_inception = InceptionV3(
    weights='imagenet',
    include_top=False,
    input_shape=(*IMAGE_SIZE, 3)
)
base_model_inception.trainable = False

model_inception = Sequential([
    base_model_inception,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

model_inception.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy']
)

model_inception.summary()

In [ ]:
print('\n' + '='*60)
print('Training Model 3: InceptionV3')
print('='*60)

history_inception = model_inception.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    class_weight=class_weights_dict,
    callbacks=get_callbacks('inception'),
    verbose=1
)

plot_history(history_inception, 'InceptionV3')

## 9. Evaluasi Individual Models

In [ ]:
pred_cnn = evaluate_model(model_cnn, val_generator, 'Custom CNN')
pred_resnet = evaluate_model(model_resnet, val_generator, 'ResNet50')
pred_inception = evaluate_model(model_inception, val_generator, 'InceptionV3')

## 10. Ensemble Evaluation (Soft Voting)

In [ ]:
print('\n' + '='*60)
print('Evaluasi: ENSEMBLE (Soft Voting)')
print('='*60)

# Prediksi probabilitas dari masing-masing model
prob_cnn = model_cnn.predict(val_generator, verbose=0)
prob_resnet = model_resnet.predict(val_generator, verbose=0)
prob_inception = model_inception.predict(val_generator, verbose=0)

# Soft Voting: rata-rata probabilitas
ensemble_prob = (prob_cnn + prob_resnet + prob_inception) / 3
ensemble_pred = np.argmax(ensemble_prob, axis=1)

y_true = val_generator.classes

print('\nClassification Report — Ensemble:')
print(classification_report(y_true, ensemble_pred, target_names=labels))

# Confusion Matrix
cm = confusion_matrix(y_true, ensemble_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix — Ensemble (Soft Voting)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix_ensemble.png'), dpi=150)
plt.show()

In [ ]:
# Perbandingan akurasi
from sklearn.metrics import accuracy_score

acc_cnn = accuracy_score(y_true, pred_cnn)
acc_resnet = accuracy_score(y_true, pred_resnet)
acc_inception = accuracy_score(y_true, pred_inception)
acc_ensemble = accuracy_score(y_true, ensemble_pred)

comparison = {
    'Custom CNN': acc_cnn,
    'ResNet50': acc_resnet,
    'InceptionV3': acc_inception,
    'Ensemble (Soft Voting)': acc_ensemble
}

print('\n' + '='*60)
print('PERBANDINGAN AKURASI')
print('='*60)
for name, acc in comparison.items():
    marker = ' ← BEST' if acc == max(comparison.values()) else ''
    print(f'  {name:30s}: {acc*100:.2f}%{marker}')

# Bar chart
plt.figure(figsize=(10, 5))
colors = ['#3498db', '#e74c3c', '#f39c12', '#2ecc71']
bars = plt.bar(comparison.keys(), [v*100 for v in comparison.values()], color=colors)
plt.ylabel('Accuracy (%)')
plt.title('Model Comparison — Validation Accuracy')
plt.ylim(0, 100)
for bar, val in zip(bars, comparison.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val*100:.1f}%', ha='center', fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison.png'), dpi=150)
plt.show()

## 11. Simpan Semua Model & Config

In [ ]:
# Simpan model final
model_cnn.save(os.path.join(OUTPUT_DIR, 'model_cnn.keras'))
model_resnet.save(os.path.join(OUTPUT_DIR, 'model_resnet.keras'))
model_inception.save(os.path.join(OUTPUT_DIR, 'model_inception.keras'))

# Simpan labels
with open(os.path.join(OUTPUT_DIR, 'labels.json'), 'w') as f:
    json.dump(labels, f, indent=2)

# Simpan ensemble config
ensemble_config = {
    'models': [
        {'name': 'Custom CNN', 'file': 'model_cnn.keras'},
        {'name': 'ResNet50', 'file': 'model_resnet.keras'},
        {'name': 'InceptionV3', 'file': 'model_inception.keras'},
    ],
    'input_size': IMAGE_SIZE[0],
    'num_classes': NUM_CLASSES,
    'voting': 'soft',
    'accuracy': {
        'cnn': float(acc_cnn),
        'resnet': float(acc_resnet),
        'inception': float(acc_inception),
        'ensemble': float(acc_ensemble),
    }
}

with open(os.path.join(OUTPUT_DIR, 'ensemble_config.json'), 'w') as f:
    json.dump(ensemble_config, f, indent=2)

# Simpan evaluation report
report = classification_report(y_true, ensemble_pred, target_names=labels)
with open(os.path.join(OUTPUT_DIR, 'evaluation_report.txt'), 'w') as f:
    f.write('Ensemble (Soft Voting) Classification Report\n')
    f.write('=' * 60 + '\n\n')
    f.write(report)

print('\n✅ Semua file tersimpan di:', OUTPUT_DIR)
print('\nFile yang perlu didownload:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f} ({size/1024/1024:.1f} MB)' if size > 1024*1024 else f'  {f} ({size/1024:.1f} KB)')

## 12. Download Artifacts

**Option 1:** Copy ke Google Drive (recommended, lebih stabil)

**Option 2:** Download langsung sebagai zip

In [ ]:
# Option 1: Copy ke Google Drive (recommended)
import shutil

DRIVE_OUTPUT = '/content/drive/MyDrive/banana-disease-artifacts'
if os.path.exists(DRIVE_OUTPUT):
    shutil.rmtree(DRIVE_OUTPUT)
shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT)
print(f'✅ Artifacts copied to Google Drive: {DRIVE_OUTPUT}')

In [ ]:
# Option 2: Download langsung ke komputer (zip)
from google.colab import files

shutil.make_archive('/content/artifacts_download', 'zip', OUTPUT_DIR)
files.download('/content/artifacts_download.zip')
print('✅ Download started!')

## 13. Test Prediksi Ensemble

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

def predict_with_voting(image_path):
    """Prediksi dengan soft voting ensemble."""
    img = load_img(image_path, target_size=IMAGE_SIZE)
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    pred_cnn = model_cnn.predict(img_array, verbose=0)
    pred_resnet = model_resnet.predict(img_array, verbose=0)
    pred_inception = model_inception.predict(img_array, verbose=0)

    # Soft voting
    final_pred = (pred_cnn + pred_resnet + pred_inception) / 3
    predicted_class = np.argmax(final_pred, axis=1)[0]
    confidence = float(final_pred[0][predicted_class])

    # Top 3
    top3_indices = np.argsort(final_pred[0])[::-1][:3]
    top3 = [(labels[i], float(final_pred[0][i])) for i in top3_indices]

    return {
        'label': labels[predicted_class],
        'confidence': confidence,
        'top3': top3
    }

# Test dengan sample dari validation set
test_images, test_labels = next(val_generator)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, ax in enumerate(axes):
    if i >= len(test_images):
        break
    ax.imshow(test_images[i])
    true_label = labels[np.argmax(test_labels[i])]

    img_batch = np.expand_dims(test_images[i], axis=0)
    p1 = model_cnn.predict(img_batch, verbose=0)
    p2 = model_resnet.predict(img_batch, verbose=0)
    p3 = model_inception.predict(img_batch, verbose=0)
    ensemble = (p1 + p2 + p3) / 3
    pred_label = labels[np.argmax(ensemble)]
    conf = float(np.max(ensemble))

    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label} ({conf:.0%})',
                 fontsize=7, color=color)
    ax.axis('off')

plt.suptitle('Ensemble Predictions on Validation Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()